In [1]:
import os

DATA_DIR = "../datasets"
print(os.listdir(DATA_DIR))

['advocate_details.csv', 'cases.csv', 'test_queries.csv']


In [2]:
import pandas as pd

DATA_DIR = "../datasets"

cases = pd.read_csv(f"{DATA_DIR}/cases.csv")
lawyers = pd.read_csv(f"{DATA_DIR}/advocate_details.csv")
queries = pd.read_csv(f"{DATA_DIR}/test_queries.csv")

print("Cases:", cases.shape)
print("Lawyers:", lawyers.shape)
print("Test queries:", queries.shape)

cases.head()

Cases: (10000, 31)
Lawyers: (7000, 44)
Test queries: (666, 6)


,case_id,cnr_number,case_number,case_type,legal_domain,court_name,court_code,court_level,court_city,filing_date,...,case_facts_summary,relief_sought,judgment_summary,evidence_type,precedent_type,appeal_available,public_interest,legal_aid_relevant,search_keywords,record_status
0,CASE-000001,K1ZXDAN0JSL48O9C,CS/47924/2019,Motor Accident Compensation Claim,Motor Accident Claims,Orissa High Court,ORIHC,High Court,Cuttack,2019-04-11,...,Synthetic case involving a motor accident comp...,"Appropriate statutory relief, injunction, comp...","The court considered the pleadings, evidence a...",Financial records,Statutory interpretation,Depends on statute/order,No,Yes,"Motor Accident Claims, Motor Accident Compensa...",SYNTHETIC - NOT AN ACTUAL CASE
1,CASE-000002,7ANB1M8N15IU8SMJ,CRL/66332/2023,POCSO / Child Protection Proceeding,Child Protection,"District Court, Hyderabad",DCHY,District Court,Hyderabad,2023-05-22,...,Synthetic case involving a pocso / child prote...,"Appropriate statutory relief, injunction, comp...","The court considered the pleadings, evidence a...",Electronic evidence,Procedural issue,Yes,No,Yes,"Child Protection, POCSO / Child Protection Pro...",SYNTHETIC - NOT AN ACTUAL CASE
2,CASE-000003,A1MQJW0LC96AUYO0,WP/43924/2018,Public Interest Litigation,Human Rights & PIL,Gauhati High Court,GAUHC,High Court,Guwahati,2018-10-30,...,Synthetic case involving a public interest lit...,"Appropriate statutory relief, injunction, comp...","The court considered the pleadings, evidence a...",Electronic evidence,Statutory interpretation,Yes,No,Yes,"Human Rights & PIL, Public Interest Litigation...",SYNTHETIC - NOT AN ACTUAL CASE
3,CASE-000004,XOCDWITJFK0RORTS,CS/32839/2019,Money Laundering / Corruption Case,White Collar Crime,"District Court, Chennai",DCCN,District Court,Chennai,2019-08-16,...,Synthetic case involving a money laundering / ...,"Appropriate statutory relief, injunction, comp...","The court considered the pleadings, evidence a...",Medical records,Statutory interpretation,Depends on statute/order,No,No,"White Collar Crime, Money Laundering / Corrupt...",SYNTHETIC - NOT AN ACTUAL CASE
4,CASE-000005,G8ILG9HPQDDA76TG,CS/75837/2020,Divorce Petition,Divorce & Matrimonial,"District Court, Pune",DCPU,District Court,Pune,2020-08-25,...,Synthetic case involving a divorce petition. T...,"Appropriate statutory relief, injunction, comp...","The court considered the pleadings, evidence a...",Financial records,Procedural issue,No,Yes,Yes,"Divorce & Matrimonial, Divorce Petition, Hindu...",SYNTHETIC - NOT AN ACTUAL CASE


In [3]:
print("Null counts (cases):\n", cases.isnull().sum())
print("\nNull counts (lawyers):\n", lawyers.isnull().sum())
print("\nDuplicate case rows:", cases.duplicated().sum())
print("Duplicate lawyer rows:", lawyers.duplicated().sum())

Null counts (cases):
 case_id                       0
cnr_number                    0
case_number                   0
case_type                     0
legal_domain                  0
court_name                    0
court_code                    0
court_level                   0
court_city                    0
filing_date                   0
filing_year                   0
judgment_date              4231
case_status                   0
case_outcome                  0
petitioner_appellant          0
respondent                    0
advocate_for_petitioner       0
advocate_for_respondent       0
primary_act                   0
sections_provisions           0
legal_issue                   0
case_facts_summary            0
relief_sought                 0
judgment_summary              0
evidence_type                 0
precedent_type                0
appeal_available              0
public_interest               0
legal_aid_relevant            0
search_keywords               0
record_status     

In [4]:
print("LAWYERS COLUMNS:")
print(lawyers.columns.tolist())
print()
print("QUERIES COLUMNS:")
print(queries.columns.tolist())

LAWYERS COLUMNS:
['record_id', 'advocate_name', 'gender', 'father_or_parent_name', 'bar_council', 'enrollment_number', 'enrollment_date', 'enrollment_year', 'advocate_status', 'designation', 'years_of_experience', 'state', 'district', 'city', 'pincode', 'primary_court', 'other_practice_courts', 'practice_area_primary', 'practice_area_secondary', 'practice_type', 'organisation_or_firm', 'bar_membership', 'law_degree', 'law_school', 'languages', 'consultation_mode', 'availability', 'email', 'phone', 'office_address', 'profile_rating', 'consultation_fee_inr', 'cases_handled_approx', 'verification_status', 'data_source', 'record_status', 'experience_band', 'career_stage', 'advocate_type', 'practice_status', 'is_fresher', 'is_experienced', 'is_senior_experienced', 'experience_level']

QUERIES COLUMNS:
['query_id', 'base_id', 'language', 'legal_domain', 'messiness_level', 'query_text']


In [5]:
lawyer_terms = set()
for col in ["practice_area_primary", "practice_area_secondary"]:
    for val in lawyers[col].dropna():
        for term in val.split(";"):
            lawyer_terms.add(term.strip())

print("Unique lawyer practice-area terms:", len(lawyer_terms))
for t in sorted(lawyer_terms):
    print(" -", t)

Unique lawyer practice-area terms: 33
 - Arbitration and Mediation
 - Banking and Finance
 - Civil Litigation
 - Commercial Law
 - Competition Law
 - Constitutional Law
 - Consumer Law
 - Copyright Law
 - Corporate Law
 - Criminal Law
 - Cyber Law
 - Data Protection and Privacy
 - Environmental Law
 - Family Law
 - GST Law
 - Human Rights Law
 - Immigration Law
 - Information Technology Law
 - Insolvency and Bankruptcy
 - Intellectual Property
 - Labour and Employment Law
 - Matrimonial Law
 - Media and Entertainment Law
 - Patent Law
 - Property Law
 - Public Interest Litigation
 - Real Estate Law
 - Securities Law
 - Securities and Capital Markets
 - Startup and Venture Capital
 - Tax Law
 - Trademark Law
 - White Collar Crime


In [6]:
# Build an explicit case-domain to lawyer-specialty taxonomy.
# Empty lists are intentional: no generic practice-area fallback is used.
lawyer_terms = set()
for col in ["practice_area_primary", "practice_area_secondary"]:
    for val in lawyers[col].dropna():
        for term in str(val).split(";"):
            lawyer_terms.add(term.strip())

case_domains = set(cases["legal_domain"].dropna().unique())

CASE_TO_LAWYER_DOMAIN = {
    "Administrative Law": ["Constitutional Law", "Public Interest Litigation"],
    "Arbitration": ["Arbitration and Mediation"],
    "Banking & Finance": ["Banking and Finance"],
    "Child Protection": ["Family Law", "Human Rights Law"],
    "Constitutional Law": ["Constitutional Law"],
    "Consumer Protection": ["Consumer Law"],
    "Contract & Agreement": ["Commercial Law", "Corporate Law"],
    "Corporate & Commercial": ["Corporate Law", "Commercial Law"],
    "Criminal Law": ["Criminal Law"],
    "Cybercrime & IT": ["Cyber Law", "Information Technology Law"],
    "Data Privacy": ["Data Protection and Privacy"],
    "Divorce & Matrimonial": ["Matrimonial Law"],
    "Education Law": [],
    "Employment & Labour": ["Labour and Employment Law"],
    "Environmental Law": ["Environmental Law"],
    "Family & Succession": ["Family Law"],
    "Human Rights & PIL": ["Human Rights Law", "Public Interest Litigation"],
    "Immigration & Citizenship": ["Immigration Law"],
    "Insolvency & Bankruptcy": ["Insolvency and Bankruptcy"],
    "Intellectual Property": ["Intellectual Property", "Patent Law", "Trademark Law", "Copyright Law"],
    "Media & Defamation": ["Media and Entertainment Law"],
    "Medical Negligence": [],
    "Motor Accident Claims": [],
    "Property & Land": ["Property Law"],
    "Real Estate & Housing": ["Real Estate Law"],
    "Tax & GST": ["Tax Law", "GST Law"],
    "White Collar Crime": ["White Collar Crime"],
}

mapped_case_domains = set(CASE_TO_LAWYER_DOMAIN)
print("Case domains missing from mapping:", case_domains - mapped_case_domains)

all_mapped_lawyer_terms = {
    term for terms in CASE_TO_LAWYER_DOMAIN.values() for term in terms
}
print("Mapped terms not found in actual lawyer vocabulary:", all_mapped_lawyer_terms - lawyer_terms)
print("Domains with no direct lawyer specialty:", [
    domain for domain, terms in CASE_TO_LAWYER_DOMAIN.items() if not terms
])

Case domains missing from mapping: set()
Mapped terms not found in actual lawyer vocabulary: set()
Domains with no direct lawyer specialty: ['Education Law', 'Medical Negligence', 'Motor Accident Claims']


In [12]:
from pathlib import Path
import sys

_search_roots = [Path.cwd(), *Path.cwd().parents]
_src_candidates = [
    root / "src" for root in _search_roots
] + [
    root / "ai-service" / "src" for root in _search_roots
] + [
    root / "copy" / "ai-service" / "src" for root in _search_roots
]
SRC_DIR = next((path for path in _src_candidates if (path / "data_loader.py").exists()), None)
if SRC_DIR is None:
    raise FileNotFoundError("Could not locate ai-service/src/data_loader.py")

sys.path.insert(0, str(SRC_DIR))
from data_loader import load_all

cases, lawyers, queries = load_all()
print("Cases:", cases.shape)
print("Lawyers:", lawyers.shape)
print("Test queries:", queries.shape)

Cases: (10000, 31)
Lawyers: (7000, 44)
Test queries: (666, 6)


In [11]:
from sentence_transformers import SentenceTransformer, util
import torch

MODELS_TO_TEST = [
    "paraphrase-multilingual-mpnet-base-v2",
    "sentence-transformers/LaBSE",
]

loaded_models = {}
for name in MODELS_TO_TEST:
    print(f"Loading {name} ...")
    loaded_models[name] = SentenceTransformer(name)
print("Done.")

c:\Users\shiva\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading paraphrase-multilingual-mpnet-base-v2 ...


c:\Users\shiva\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shiva\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading sentence-transformers/LaBSE ...


c:\Users\shiva\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shiva\.cache\huggingface\hub\models--sentence-transformers--LaBSE. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Done.


In [13]:
import random
import numpy as np
import pandas as pd

random.seed(42)

# Build sample_queries explicitly, one language at a time — avoids any
# groupby/apply edge cases with column retention
sample_parts = []
for lang in queries["language"].unique():
    subset = queries[queries["language"] == lang]
    n = min(20, len(subset))
    sample_parts.append(subset.sample(n, random_state=42))

sample_queries = pd.concat(sample_parts, ignore_index=True)

# Diagnostic — confirm the columns are what we expect before running anything expensive
print("sample_queries columns:", sample_queries.columns.tolist())
print("sample_queries shape:", sample_queries.shape)
print(sample_queries["language"].value_counts())

sample_queries columns: ['query_id', 'base_id', 'language', 'legal_domain', 'messiness_level', 'query_text']
sample_queries shape: (60, 6)
language
Hindi       20
English     20
Hinglish    20
Name: count, dtype: int64


In [14]:
import random
import numpy as np
import pandas as pd

random.seed(42)

def cos_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def evaluate_model(model):
    results = []

    for _, row in sample_queries.iterrows():
        query_text = row["query_text"]
        true_domain = row["legal_domain"].split(" / ")[0]  # primary domain for ambiguous rows

        true_pool = cases[cases["legal_domain"] == true_domain]["case_facts_summary"].dropna()
        true_domain_cases = true_pool.sample(min(3, len(true_pool)), random_state=42).tolist()

        wrong_domain = random.choice([d for d in cases["legal_domain"].unique() if d != true_domain])
        wrong_pool = cases[cases["legal_domain"] == wrong_domain]["case_facts_summary"].dropna()
        wrong_domain_cases = wrong_pool.sample(min(3, len(wrong_pool)), random_state=42).tolist()

        q_emb = model.encode(query_text, convert_to_numpy=True)
        true_embs = model.encode(true_domain_cases, convert_to_numpy=True)
        wrong_embs = model.encode(wrong_domain_cases, convert_to_numpy=True)

        true_sim = np.mean([cos_sim(q_emb, e) for e in true_embs])
        wrong_sim = np.mean([cos_sim(q_emb, e) for e in wrong_embs])

        results.append({
            "language": row["language"],
            "true_sim": true_sim,
            "wrong_sim": wrong_sim,
            "gap": true_sim - wrong_sim,
        })

    return pd.DataFrame(results)

bake_off_results = {}
for name, model in loaded_models.items():
    print(f"Evaluating {name} ...")
    bake_off_results[name] = evaluate_model(model)
    df = bake_off_results[name]
    print(f"  Overall avg gap (true_sim - wrong_sim): {df['gap'].mean():.4f}")
    print(f"  By language:\n{df.groupby('language')['gap'].mean()}\n")

Evaluating paraphrase-multilingual-mpnet-base-v2 ...
  Overall avg gap (true_sim - wrong_sim): 0.1221
  By language:
language
English     0.163816
Hindi       0.110430
Hinglish    0.091969
Name: gap, dtype: float32

Evaluating sentence-transformers/LaBSE ...
  Overall avg gap (true_sim - wrong_sim): 0.0856
  By language:
language
English     0.076326
Hindi       0.080611
Hinglish    0.099738
Name: gap, dtype: float32



In [15]:
import sys
sys.path.append("../src")
from preprocessing import detect_language, normalize_text

queries["detected_language"] = queries["query_text"].apply(detect_language)
accuracy = (queries["detected_language"] == queries["language"]).mean()
print(f"Language detection accuracy: {accuracy:.2%}")

Language detection accuracy: 92.34%


In [16]:
import numpy as np
import time

model = loaded_models["paraphrase-multilingual-mpnet-base-v2"]

# Use case_facts_summary as the primary text representation of each case.
# Drop rows with missing text — can't embed nothing.
cases_to_embed = cases.dropna(subset=["case_facts_summary"]).reset_index(drop=True)
print(f"Embedding {len(cases_to_embed)} cases (dropped {len(cases) - len(cases_to_embed)} with missing text)...")

texts = cases_to_embed["case_facts_summary"].tolist()

start = time.time()
case_embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)
elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s — embeddings shape: {case_embeddings.shape}")

Embedding 10000 cases (dropped 0 with missing text)...


Batches: 100%|██████████| 157/157 [03:41<00:00,  1.41s/it]

Done in 221.2s — embeddings shape: (10000, 768)


In [17]:
import os

os.makedirs("../data/processed", exist_ok=True)

np.save("../data/processed/case_embeddings.npy", case_embeddings)
cases_to_embed[["case_id", "legal_domain", "case_type"]].to_csv(
    "../data/processed/case_embeddings_index.csv", index=False
)

print("Saved embeddings and index mapping.")

Saved embeddings and index mapping.


In [18]:
import pandas as pd
import numpy as np
from collections import Counter

# --- Rebuild everything Phase E depends on, in one place ---

model = loaded_models["paraphrase-multilingual-mpnet-base-v2"]

case_embeddings = np.load("../data/processed/case_embeddings.npy")
cases_to_embed = cases.dropna(subset=["case_facts_summary"]).reset_index(drop=True)
assert len(cases_to_embed) == len(case_embeddings), "Mismatch — rerun Phase C"

def retrieve_similar_cases_local(query_text, model, top_k=5):
    query_embedding = model.encode(query_text, convert_to_numpy=True)
    sims = case_embeddings @ query_embedding / (
        np.linalg.norm(case_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )
    top_idx = np.argsort(sims)[::-1][:top_k]
    result = cases_to_embed.iloc[top_idx][["case_id", "legal_domain", "case_type", "case_facts_summary"]].copy()
    result["score"] = sims[top_idx]
    return result

def predict_domains_from_retrieval(query_text, top_k=5, max_domains=2, min_confidence=0.20):
    """Return confidence-ranked domains from similarity-weighted case votes."""
    results = retrieve_similar_cases_local(query_text, model, top_k=top_k)
    weights = results["score"].clip(lower=0)
    votes = results.assign(weight=weights).groupby("legal_domain")["weight"].sum()
    if votes.sum() == 0:
        return []
    confidence = (votes / votes.sum()).sort_values(ascending=False)
    selected = confidence[confidence >= min_confidence].head(max_domains)
    if selected.empty:
        selected = confidence.head(1)
    return [(domain, float(score)) for domain, score in selected.items()]

def predict_domain_from_retrieval(query_text, top_k=5):
    """Compatibility helper returning the highest-confidence domain and retrieved domains."""
    predictions = predict_domains_from_retrieval(query_text, top_k=top_k)
    results = retrieve_similar_cases_local(query_text, model, top_k=top_k)
    predicted = predictions[0][0] if predictions else None
    return predicted, results["legal_domain"].tolist()

print("Setup complete.")
print("case_embeddings:", case_embeddings.shape)
print("cases_to_embed:", cases_to_embed.shape)

Setup complete.
case_embeddings: (10000, 768)
cases_to_embed: (10000, 31)


In [19]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient
import numpy as np
import pandas as pd

load_dotenv("../.env")

#client = MongoClient(os.getenv("MONGO_URI"))

#db = client["askvocate"]
#cases_collection = db["cases"]

# Load what Phase C saved
case_embeddings = np.load("../data/processed/case_embeddings.npy")
embed_index = pd.read_csv("../data/processed/case_embeddings_index.csv")

# Rebuild the full case rows (with embeddings attached) to insert
cases_to_embed = cases.dropna(subset=["case_facts_summary"]).reset_index(drop=True)
assert len(cases_to_embed) == len(case_embeddings), "Mismatch between cases and embeddings — rerun Phase C"

docs = []
for i, row in cases_to_embed.iterrows():
    doc = row.to_dict()
    doc["embedding"] = case_embeddings[i].tolist()  # MongoDB needs a plain list, not numpy array
    docs.append(doc)

#cases_collection.delete_many({})  # clear if rerunning, avoid duplicates
#cases_collection.insert_many(docs)
#print(f"Inserted {len(docs)} case documents into MongoDB.")

In [ ]:
import pandas as pd
from collections import Counter

model = loaded_models["paraphrase-multilingual-mpnet-base-v2"]

def predict_domain_from_retrieval(query_text, top_k=5):
    """Retrieve top-k similar cases, predict domain via majority vote."""
    results = retrieve_similar_cases_local(query_text, model, top_k=top_k)
    domain_votes = Counter(results["legal_domain"])
    predicted = domain_votes.most_common(1)[0][0]
    return predicted, results["legal_domain"].tolist()

eval_results = []

for idx, row in queries.iterrows():
    true_domains = [d.strip() for d in str(row["legal_domain"]).split(" / ")]  # handles ambiguous rows
    predicted, top5_domains = predict_domain_from_retrieval(row["query_text"], top_k=5)

    top1_hit = predicted in true_domains
    top5_hit = any(d in true_domains for d in top5_domains)

    eval_results.append({
        "query_id": row["query_id"],
        "language": row["language"],
        "messiness_level": row["messiness_level"],
        "true_domain": row["legal_domain"],
        "predicted_domain": predicted,
        "top1_hit": int(top1_hit),
        "top5_hit": int(top5_hit),
    })

    if idx % 100 == 0:
        print(f"Processed {idx}/{len(queries)}...")

eval_df = pd.DataFrame(eval_results)
print("\nDone.")

Processed 0/666...
Processed 100/666...
Processed 200/666...
Processed 300/666...
Processed 400/666...
Processed 500/666...
Processed 600/666...

Done.


In [24]:
from pathlib import Path

if "eval_df" not in globals():
    search_roots = [Path.cwd(), *Path.cwd().parents]
    evaluation_candidates = [
        root / "data" / "processed" / "phase_e_evaluation_results.csv"
        for root in search_roots
    ] + [
        root / "ai-service" / "data" / "processed" / "phase_e_evaluation_results.csv"
        for root in search_roots
    ] + [
        root / "copy" / "ai-service" / "data" / "processed" / "phase_e_evaluation_results.csv"
        for root in search_roots
    ]
    evaluation_file = next((path for path in evaluation_candidates if path.exists()), None)
    if evaluation_file is None:
        raise RuntimeError(
            "eval_df is not available. Run the evaluation cell above first to create it."
        )
    eval_df = pd.read_csv(evaluation_file)
    print(f"Loaded saved evaluation results from: {evaluation_file}")

print("=== OVERALL ===")
print(f"Top-1 Accuracy: {eval_df['top1_hit'].mean()*100:.1f}%")
print(f"Top-5 Recall:   {eval_df['top5_hit'].mean()*100:.1f}%")

print("\n=== BY LANGUAGE ===")
print(eval_df.groupby("language")[["top1_hit", "top5_hit"]].mean() * 100)

print("\n=== BY MESSINESS LEVEL ===")
print(eval_df.groupby("messiness_level")[["top1_hit", "top5_hit"]].mean() * 100)

print("\n=== WORST-PERFORMING DOMAINS (Top-1) ===")
domain_acc = eval_df.groupby("true_domain")["top1_hit"].mean().sort_values()
print(domain_acc.head(10) * 100)

=== OVERALL ===
Top-1 Accuracy: 61.7%
Top-5 Recall:   61.7%

=== BY LANGUAGE ===
           top1_hit   top5_hit
language                      
English   74.796748  74.796748
Hindi     58.823529  58.823529
Hinglish  58.702065  58.702065

=== BY MESSINESS LEVEL ===
                  top1_hit   top5_hit
messiness_level                      
ambiguous        88.888889  88.888889
clean            64.814815  64.814815
heavy            53.439153  53.439153
light            62.962963  62.962963
medium           61.111111  61.111111

=== WORST-PERFORMING DOMAINS (Top-1) ===
true_domain
Criminal Law / White Collar Crime     0.000000
Human Rights & PIL                    0.000000
Media & Defamation                    8.695652
Contract & Agreement                  8.695652
Consumer Protection                  13.043478
Constitutional Law                   21.739130
Property & Land                      26.086957
Administrative Law                   30.434783
Employment & Labour                  34.

In [25]:
# Check: how many *distinct* domains show up in the top-5 retrieved cases, on average?
diversity_check = []
for idx, row in queries.sample(30, random_state=1).iterrows():
    _, top5_domains = predict_domain_from_retrieval(row["query_text"], top_k=5)
    diversity_check.append(len(set(top5_domains)))

print("Avg distinct domains among top-5 retrieved cases:", sum(diversity_check) / len(diversity_check))
print("Distribution:", pd.Series(diversity_check).value_counts().sort_index())

Avg distinct domains among top-5 retrieved cases: 1.0
Distribution: 1    30
Name: count, dtype: int64


In [26]:
print(queries[queries["legal_domain"].str.contains("Human Rights", na=False)]["legal_domain"].count())

26


In [27]:
eval_df.to_csv("../data/processed/phase_e_evaluation_results.csv", index=False)
print("Saved evaluation results.")

Saved evaluation results.


In [ ]:
hr_queries = queries[queries["legal_domain"].str.contains("Human Rights", na=False)]

confusion = []
for idx, row in hr_queries.iterrows():
    predicted, _ = predict_domain_from_retrieval(row["query_text"], top_k=5)
    confusion.append(predicted)

print("What Human Rights & PIL queries actually get predicted as:")
print(pd.Series(confusion).value_counts())

What Human Rights & PIL queries actually get predicted as:
Environmental Law     16
Criminal Law           5
Child Protection       4
Constitutional Law     1
Name: count, dtype: int64


In [ ]:
# Build enriched text using the same fields used by the case search profile.
cases_to_embed_v2 = cases.dropna(subset=["case_facts_summary"]).reset_index(drop=True)
cases_to_embed_v2["combo_text"] = (
    cases_to_embed_v2["legal_issue"].fillna("") + " " +
    cases_to_embed_v2["case_facts_summary"].fillna("") + " " +
    cases_to_embed_v2["search_keywords"].fillna("")
)

print("Re-embedding with enriched text...")
case_embeddings_v2 = model.encode(
    cases_to_embed_v2["combo_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

def retrieve_similar_cases_v2(query_text, model, top_k=5):
    query_embedding = model.encode(query_text, convert_to_numpy=True)
    sims = case_embeddings_v2 @ query_embedding / (
        np.linalg.norm(case_embeddings_v2, axis=1) * np.linalg.norm(query_embedding)
    )
    top_idx = np.argsort(sims)[::-1][:top_k]
    result = cases_to_embed_v2.iloc[top_idx][
        ["case_id", "legal_domain", "case_facts_summary"]
    ].copy()
    result["score"] = sims[top_idx]
    return result

def predict_domains_v2(query_text, top_k=5, max_domains=2, min_confidence=0.20):
    """Return confidence-ranked legal domains without collapsing clear ambiguity."""
    results = retrieve_similar_cases_v2(query_text, model, top_k=top_k)
    weights = results["score"].clip(lower=0)
    votes = results.assign(weight=weights).groupby("legal_domain")["weight"].sum()
    if votes.sum() == 0:
        return []
    confidence = (votes / votes.sum()).sort_values(ascending=False)
    selected = confidence[confidence >= min_confidence].head(max_domains)
    if selected.empty:
        selected = confidence.head(1)
    return [(domain, float(score)) for domain, score in selected.items()]

def predict_domain_v2(query_text, top_k=5):
    """Return the highest-confidence domain for legacy evaluation output."""
    predictions = predict_domains_v2(query_text, top_k=top_k)
    return predictions[0][0] if predictions else None

print("v2 setup complete.")
print("case_embeddings_v2:", case_embeddings_v2.shape)
print("cases_to_embed_v2:", cases_to_embed_v2.shape)

Re-embedding with enriched text...


Batches:  17%|█▋        | 27/157 [01:46<08:20,  3.85s/it]

In [ ]:
np.save("../data/processed/case_embeddings.npy", case_embeddings_v2)
cases_to_embed_v2[["case_id", "legal_domain", "case_type", "combo_text"]].to_csv(
    "../data/processed/case_embeddings_index.csv", index=False
)


case_embeddings = case_embeddings_v2
cases_to_embed = cases_to_embed_v2
retrieve_similar_cases_local = retrieve_similar_cases_v2

eval_df_v2.to_csv("../data/processed/phase_e_evaluation_results_v2.csv", index=False)
print("v2 locked in as production pipeline. Files saved.")

v2 locked in as production pipeline. Files saved.


In [ ]:
def recommend_lawyers(user_prompt, user_city, user_state=None, top_k=None, secondary_weight=0.65):
    """Match available lawyers to one or more predicted domains in one city.

    No specialization, geographic, or national fallback is used. An empty
    result means no verified specialist matched the requested constraints.
    """
    predicted_domains = predict_domains_v2(user_prompt, top_k=5, max_domains=2)
    domain_names = [domain for domain, _ in predicted_domains]
    target_terms = {
        term
        for domain in domain_names
        for term in CASE_TO_LAWYER_DOMAIN.get(domain, [])
    }

    result_columns = [
        "advocate_name", "practice_area_primary", "practice_area_secondary", "match_tier",
        "consultation_fee_inr", "profile_rating", "years_of_experience",
        "city", "state", "availability", "ranking_score",
        "score_specialization", "score_domain_coverage", "score_affordability",
        "score_rating", "score_experience",
    ]
    if not target_terms:
        return predicted_domains, lawyers.iloc[0:0].reindex(columns=result_columns)

    def normalise(value):
        return " ".join(str(value).casefold().split())

    candidates = lawyers[
        lawyers["city"].map(normalise).eq(normalise(user_city))
        & lawyers["availability"].ne("Not accepting new matters")
    ].copy()
    if user_state is not None:
        candidates = candidates[
            candidates["state"].map(normalise).eq(normalise(user_state))
        ]
    if candidates.empty:
        return predicted_domains, lawyers.iloc[0:0].reindex(columns=result_columns)

    normalised_terms = {normalise(term) for term in target_terms}
    candidates["_primary"] = candidates["practice_area_primary"].map(normalise)
    candidates["_secondary"] = candidates["practice_area_secondary"].map(
        lambda value: {normalise(term) for term in str(value).split(";")}
    )
    candidates["_primary_match"] = candidates["_primary"].isin(normalised_terms)
    candidates["_secondary_match"] = candidates["_secondary"].map(
        lambda terms: bool(terms & normalised_terms)
    )
    candidates = candidates[
        candidates["_primary_match"] | candidates["_secondary_match"]
    ].copy()
    if candidates.empty:
        return predicted_domains, lawyers.iloc[0:0].reindex(columns=result_columns)

    candidates["match_tier"] = candidates["_primary_match"].map(
        {True: "primary", False: "secondary"}
    )
    candidates["score_specialization"] = candidates["_primary_match"].where(
        candidates["_primary_match"], secondary_weight
    )
    candidates["score_domain_coverage"] = candidates.apply(
        lambda row: sum(
            normalise(term) == row["_primary"] or normalise(term) in row["_secondary"]
            for domain in domain_names
            for term in CASE_TO_LAWYER_DOMAIN.get(domain, [])
        ),
        axis=1,
    )
    fees = pd.to_numeric(lawyers["consultation_fee_inr"])
    fee_min, fee_max = fees.min(), fees.max()
    candidates["score_affordability"] = 1.0 if fee_max == fee_min else 1.0 - (
        pd.to_numeric(candidates["consultation_fee_inr"]) - fee_min
    ) / (fee_max - fee_min)
    candidates["score_rating"] = pd.to_numeric(candidates["profile_rating"]).clip(0, 5) / 5
    candidates["score_experience"] = (
        pd.to_numeric(candidates["years_of_experience"]).clip(0, 25) / 25
    )
    candidates["ranking_score"] = (
        0.40 * candidates["score_specialization"]
        + 0.15 * candidates["score_domain_coverage"].clip(upper=2) / 2
        + 0.20 * candidates["score_affordability"]
        + 0.15 * candidates["score_rating"]
        + 0.10 * candidates["score_experience"]
    )
    candidates = candidates.sort_values(
        ["ranking_score", "consultation_fee_inr"], ascending=[False, True]
    )
    candidates = candidates.drop(
        columns=["_primary", "_secondary", "_primary_match", "_secondary_match"]
    )
    if top_k is not None:
        candidates = candidates.head(top_k)
    return predicted_domains, candidates[result_columns]

In [ ]:
predicted_domains, top_lawyers = recommend_lawyers(
    user_prompt="mera landlord security deposit wapas nahi de raha",
    user_city="Delhi",
    user_state="Delhi",
    top_k=5,
)
print(f"Predicted domains: {predicted_domains}")
top_lawyers

Predicted domain: Real Estate & Housing


,advocate_name,practice_area_primary,practice_area_secondary,match_tier,consultation_fee_inr,profile_rating,years_of_experience,city,state,availability,ranking_score,score_specialization,score_affordability,score_rating,score_experience
4892,Adv. Tanvi Verma,Real Estate Law,Consumer Law; Criminal Law,primary,1500,4.9,30,New Delhi,Delhi,Available for new matters,0.960286,1.0,0.857143,0.98,1.00
3274,Adv. Simran Malhotra,Real Estate Law,Commercial Law; GST Law,primary,500,4.2,23,New Delhi,Delhi,Limited availability,0.956000,1.0,1.000000,0.84,0.92
4792,Adv. Nidhi Reddy,Real Estate Law,Criminal Law; Civil Litigation,primary,1000,3.6,31,New Delhi,Delhi,Available for new matters,0.926143,1.0,0.928571,0.72,1.00
254,Adv. Aadhya Ghosh,Real Estate Law,Property Law; Labour and Employment Law,primary,750,4.9,5,New Delhi,Delhi,Limited availability,0.867071,1.0,0.964286,0.98,0.20
4589,Adv. Nidhi Chopra,Real Estate Law,Competition Law; Immigration Law,primary,750,3.5,9,New Delhi,Delhi,Available for consultation,0.835071,1.0,0.964286,0.70,0.36


In [ ]:
from anthropic import Anthropic
import os
from dotenv import load_dotenv

load_dotenv("../.env")
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def generate_guidance(user_prompt, predicted_domain, top_lawyers_df, similar_cases_df):
    case_context = "\n".join(
        f"- {row['legal_domain']}: {row['case_facts_summary'][:150]}..."
        for _, row in similar_cases_df.head(3).iterrows()
    )

    lawyer_names = ", ".join(top_lawyers_df["advocate_name"].head(3).tolist())

    system_prompt = (
        "You are a plain-language legal orientation assistant for Askvocate, an Indian legal "
        "platform. You are NOT a lawyer and must never give specific legal advice, predict case "
        "outcomes, or cite exact sections/acts as certain. Your only job: briefly explain what "
        "legal area the user's situation falls under, in simple language, and point them toward "
        "consulting a lawyer. Keep it to 3-4 sentences. Respond in the same language style the "
        "user wrote in (Hindi/English/Hinglish)."
    )

    user_message = f"""User's situation: {user_prompt}

Detected legal domain: {predicted_domain}

Similar past cases in this domain:
{case_context}

Write a brief, warm, plain-language explanation of what kind of legal matter this is and why they should speak with a lawyer specializing in {predicted_domain}."""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=300,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text


def full_pipeline(user_prompt, user_city, user_state=None, top_k=5):
    """End-to-end: domain prediction -> lawyer ranking -> LLM guidance."""
    predicted_domain, top_lawyers = recommend_lawyers(user_prompt, user_city, user_state, top_k=top_k)
    similar_cases = retrieve_similar_cases_v2(user_prompt, model, top_k=5)
    guidance = generate_guidance(user_prompt, predicted_domain, top_lawyers, similar_cases)

    return {
        "predicted_domain": predicted_domain,
        "guidance": guidance,
        "recommended_lawyers": top_lawyers.to_dict(orient="records"),
    }

ModuleNotFoundError: No module named 'anthropic'

In [ ]:
result = full_pipeline(
    user_prompt="mera landlord security deposit wapas nahi de raha",
    user_city="Delhi",
    user_state="Delhi",
    top_k=3
)
print(result["predicted_domain"])
print()
print(result["guidance"])
print()
for l in result["recommended_lawyers"]:
    print(l["advocate_name"], "-", l["consultation_fee_inr"])